# Установка зависимостей и проверка работы моделей(выполнять только один раз)

In [ ]:
!pip install -U transformers datasets accelerate evaluate scikit-learn pandas numpy

In [ ]:
!pip uninstall -y transformers
!pip install "transformers==4.43.3"

Found existing installation: transformers 5.4.0
Uninstalling transformers-5.4.0:
  Successfully uninstalled transformers-5.4.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 81.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires p

# Генерация train_augmentation.csv

In [ ]:

import os
import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import evaluate

from google.colab import drive
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed
)
from transformers.trainer_utils import get_last_checkpoint

os.environ["SAFETENSORS_FAST_GPU"] = "0"

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def set_seed(seed):
   random.seed(seed)
   np.random.seed(seed)
   torch.manual_seed(seed)
   torch.cuda.manual_seed_all(seed)

SEED = 42
set_seed(SEED)

device: cuda


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
drive_root = '/content/drive/MyDrive/papadyk-collab/vkr'

test_path = os.path.join(drive_root, 'test.csv')
BASE_DIR = os.path.join(drive_root, 'output')

TRAIN_FILE = os.path.join(drive_root, 'train_augmented.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

from huggingface_hub import login
login(HF_TOKEN)

In [ ]:
# ============================================================
# 1. CONFIG
# ============================================================
MODEL_NAME = "DeepPavlov/rubert-base-cased"

VAL_SIZE = 0.2
MAX_LENGTH = 256

BATCH_SIZE = 16
NUM_EPOCHS = 10
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

FORCE_RESTART_FROM_SCRATCH = False
SAVE_TOTAL_LIMIT = 2

# Если хочешь фиксировать один и тот же split между перезапусками
USE_SAVED_SPLIT_IF_EXISTS = True


# ============================================================
# 4. OUTPUT PATHS DEPEND ON TRAIN FILE NAME
# ============================================================
train_stem = Path(TRAIN_FILE).stem
safe_train_stem = re.sub(r"[^a-zA-Z0-9а-яА-Я_-]+", "_", train_stem)
model_tag = MODEL_NAME.split("/")[-1]

RUN_NAME = f"{safe_train_stem}__{model_tag}"
OUTPUT_DIR = f"{BASE_DIR}/trainer_output"
BEST_MODEL_DIR = f"{BASE_DIR}/best_model"
REPORTS_DIR = f"{BASE_DIR}/reports"

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print("RUN_NAME:", RUN_NAME)
print("BASE_DIR:", BASE_DIR)


# ============================================================
# 5. LOAD DATA
# ============================================================
df = pd.read_csv(TRAIN_FILE)
print("Raw shape:", df.shape)
print("Columns:", df.columns.tolist())

text_col = "text"
label_col = "label"

df = df[[text_col, label_col]].copy()
df = df.dropna()
df[text_col] = df[text_col].astype(str).str.strip()
df[label_col] = df[label_col].astype(str).str.strip()
df = df[(df[text_col] != "") & (df[label_col] != "")].reset_index(drop=True)

print("After cleanup:", df.shape)
print(df[label_col].value_counts())


# ============================================================
# 6. LABEL ENCODING
# ============================================================
labels = sorted(df[label_col].unique().tolist())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

df["label_id"] = df[label_col].map(label2id)

with open(f"{REPORTS_DIR}/label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "label2id": label2id,
            "id2label": {str(k): v for k, v in id2label.items()}
        },
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 7. TRAIN / VALID SPLIT
#    If saved split exists, reuse it for reproducibility
# ============================================================
train_split_path = f"{REPORTS_DIR}/train_split.csv"
valid_split_path = f"{REPORTS_DIR}/valid_split.csv"

if USE_SAVED_SPLIT_IF_EXISTS and os.path.exists(train_split_path) and os.path.exists(valid_split_path):
    print("Using saved train/valid split from disk")
    train_df = pd.read_csv(train_split_path)
    valid_df = pd.read_csv(valid_split_path)
else:
    print("Creating new stratified split")
    train_df, valid_df = train_test_split(
        df,
        test_size=VAL_SIZE,
        random_state=SEED,
        stratify=df["label_id"]
    )
    train_df = train_df.reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)

    train_df.to_csv(train_split_path, index=False)
    valid_df.to_csv(valid_split_path, index=False)

print("train:", train_df.shape)
print("valid:", valid_df.shape)


# ============================================================
# 8. HUGGING FACE DATASETS
# ============================================================
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df[[text_col, "label_id"]], preserve_index=False),
    "validation": Dataset.from_pandas(valid_df[[text_col, "label_id"]], preserve_index=False)
})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch[text_col],
        truncation=True,
        max_length=MAX_LENGTH
    )

dataset = dataset.map(tokenize_batch, batched=True)
dataset = dataset.rename_column("label_id", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


# ============================================================
# 9. METRICS
# ============================================================
def compute_metrics(eval_pred):
    logits, labels_true = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels_true, preds)
    bal_acc = balanced_accuracy_score(labels_true, preds)
    f1_macro = f1_score(labels_true, preds, average="macro", zero_division=0)
    f1_weighted = f1_score(labels_true, preds, average="weighted", zero_division=0)
    f1_micro = f1_score(labels_true, preds, average="micro", zero_division=0)

    precision_macro, recall_macro, _, _ = precision_recall_fscore_support(
        labels_true, preds, average="macro", zero_division=0
    )

    return {
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "f1_micro": f1_micro,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
    }


# ============================================================
# 10. MODEL
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

# ============================================================
# 11. CHECKPOINT DISCOVERY
# ============================================================
last_checkpoint = None
if os.path.isdir(OUTPUT_DIR):
    last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

print("Last checkpoint:", last_checkpoint)


# ============================================================
# 12. TRAINING ARGUMENTS
# ============================================================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    run_name=RUN_NAME,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=SAVE_TOTAL_LIMIT,

    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
    save_safetensors=False,
)


# ============================================================
# 13. TRAINER
# ============================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


# ============================================================
# 14. TRAIN / RESUME
# ============================================================
if FORCE_RESTART_FROM_SCRATCH:
    print("Forced restart from scratch")
    train_result = trainer.train()
else:
    if last_checkpoint is not None:
        print(f"Resume from checkpoint: {last_checkpoint}")
        train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
    else:
        print("Start training from scratch")
        train_result = trainer.train()


# ============================================================
# 15. SAVE BEST MODEL
# ============================================================
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

with open(f"{REPORTS_DIR}/train_result.json", "w", encoding="utf-8") as f:
    json.dump(train_result.metrics, f, ensure_ascii=False, indent=2)


# ============================================================
# 16. FINAL EVALUATION
# ============================================================
eval_metrics = trainer.evaluate(dataset["validation"])

with open(f"{REPORTS_DIR}/eval_metrics.json", "w", encoding="utf-8") as f:
    json.dump(eval_metrics, f, ensure_ascii=False, indent=2)


# ============================================================
# 17. VALIDATION PREDICTIONS
# ============================================================
pred_output = trainer.predict(dataset["validation"])
pred_logits = pred_output.predictions
y_true = pred_output.label_ids
y_pred = np.argmax(pred_logits, axis=1)

valid_df_result = valid_df.copy()
valid_df_result["y_true_id"] = y_true
valid_df_result["y_pred_id"] = y_pred
valid_df_result["y_true"] = [id2label[int(x)] for x in y_true]
valid_df_result["y_pred"] = [id2label[int(x)] for x in y_pred]
valid_df_result["is_correct"] = (valid_df_result["y_true_id"] == valid_df_result["y_pred_id"]).astype(int)

probs = torch.softmax(torch.tensor(pred_logits), dim=1).numpy()
valid_df_result["pred_confidence"] = probs.max(axis=1)

valid_df_result.to_csv(f"{REPORTS_DIR}/validation_predictions.csv", index=False)


# ============================================================
# 18. CLASSIFICATION REPORT + CONFUSION MATRIX
# ============================================================
report_dict = classification_report(
    y_true,
    y_pred,
    target_names=[id2label[i] for i in range(len(id2label))],
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv(f"{REPORTS_DIR}/classification_report.csv", encoding="utf-8")

with open(f"{REPORTS_DIR}/classification_report.json", "w", encoding="utf-8") as f:
    json.dump(report_dict, f, ensure_ascii=False, indent=2)

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(labels))))
cm_df = pd.DataFrame(
    cm,
    index=[f"true::{id2label[i]}" for i in range(len(labels))],
    columns=[f"pred::{id2label[i]}" for i in range(len(labels))]
)
cm_df.to_csv(f"{REPORTS_DIR}/confusion_matrix.csv", encoding="utf-8")


# ============================================================
# 19. RUN SUMMARY
# ============================================================
summary = {
    "train_file": TRAIN_FILE,
    "run_name": RUN_NAME,
    "model_name": MODEL_NAME,
    "text_col": text_col,
    "label_col": label_col,
    "num_rows_total": int(len(df)),
    "num_rows_train": int(len(train_df)),
    "num_rows_valid": int(len(valid_df)),
    "num_classes": int(len(labels)),
    "labels": labels,
    "last_checkpoint_used": last_checkpoint,
    "best_model_checkpoint": trainer.state.best_model_checkpoint,
    "best_metric": trainer.state.best_metric,
    "eval_metrics": eval_metrics,
}

with open(f"{REPORTS_DIR}/run_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\nDone. All files saved to:", BASE_DIR)

RUN_NAME: train_augmented__rubert-base-cased
BASE_DIR: /content/drive/MyDrive/papadyk-collab/vkr/output
Raw shape: (1994, 2)
Columns: ['label', 'text']
After cleanup: (1994, 2)
label
Блок технического директора                                                     202
Блок директора по мощностям                                                     197
Блок директора по строительству                                                 134
Управление по проектным работам                                                 110
Блок заместителя генерального директора по безопасности                         101
Генеральный директор                                                             83
Проект "Нефтяные краюшки"                                                        64
Блок деректора по газу                                                           59
Блок заместителя генерального директора по закупкам                              54
Блок заместителя генерального директора по организационным во

Map:   0%|          | 0/1595 [00:00<?, ? examples/s]

Map:   0%|          | 0/399 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Last checkpoint: None
Start training from scratch


Epoch,Training Loss,Validation Loss,Accuracy,Balanced Accuracy,F1 Macro,F1 Weighted,F1 Micro,Precision Macro,Recall Macro
1,3.509100,3.363408,0.107769,0.050149,0.019956,0.031860,0.107769,0.013138,0.050149
2,3.067400,2.773971,0.300752,0.239989,0.195894,0.225250,0.300752,0.270039,0.239989
3,2.487000,2.275276,0.473684,0.407417,0.367405,0.394985,0.473684,0.475764,0.407417
4,1.981100,1.968660,0.558897,0.482961,0.466121,0.500106,0.558897,0.568789,0.482961
5,1.599400,1.735258,0.614035,0.552034,0.544914,0.568700,0.614035,0.617512,0.552034
6,1.300300,1.623200,0.631579,0.576461,0.564472,0.587832,0.631579,0.618059,0.576461
7,1.085800,1.526446,0.656642,0.619022,0.616059,0.633863,0.656642,0.654435,0.619022
8,0.921200,1.467301,0.659148,0.626462,0.616916,0.630420,0.659148,0.650329,0.626462
9,0.810500,1.423855,0.674185,0.640959,0.636556,0.650021,0.674185,0.670233,0.640959
10,0.745300,1.399380,0.676692,0.648334,0.640819,0.651832,0.676692,0.669891,0.648334


{
  "train_file": "/content/drive/MyDrive/papadyk-collab/vkr/train_augmented.csv",
  "run_name": "train_augmented__rubert-base-cased",
  "model_name": "DeepPavlov/rubert-base-cased",
  "text_col": "text",
  "label_col": "label",
  "num_rows_total": 1994,
  "num_rows_train": 1595,
  "num_rows_valid": 399,
  "num_classes": 36,
  "labels": [
    "Блок бизнес-директора",
    "Блок деректора по газу",
    "Блок директора по газовым проектам",
    "Блок директора по мощностям",
    "Блок директора по персоналу",
    "Блок директора по портфелю",
    "Блок директора по проектированию",
    "Блок директора по строительству",
    "Блок заместителя генерального директора по безопасности",
    "Блок заместителя генерального директора по закупкам",
    "Блок заместителя генерального директора по защите",
    "Блок заместителя генерального директора по имуществу",
    "Блок заместителя генерального директора по организационным вопросам",
    "Блок заместителя генерального директора по строительству